# Fixed Gender Debiasing for DistilBERT using MLM

This notebook properly implements gender debiasing using Masked Language Modeling with Counterfactual Data Augmentation (CDA).

In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. Load Dataset

In [ ]:
   "source": [
    "# Load dataset (keep brackets to identify gender tokens)\n",
    "file_path = 'data/processed/dataset.json'\n",
    "\n",
    "with open(file_path, 'r') as f:\n",
    "    data = json.load(f)\n",
    "\n",
    "print(f\"Loaded {len(data)} counterfactual pairs\")\n",
    "print(f\"Example pair:\")\n",
    "print(f\"  Pro-stereotyped: {data[0]['pro']}\")\n",
    "print(f\"  Anti-stereotyped: {data[0]['anti']}\")\n",
    "print(f\"\\nNote: Words in [brackets] indicate gender-specific tokens to be masked\")"
   ]

## 2. Initialize Model and Tokenizer

In [ ]:
model_name = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMaskedLM.from_pretrained(model_name)

print(f"Mask token: {tokenizer.mask_token} (ID: {tokenizer.mask_token_id})")

## 3. Configure LoRA (FIXED)

In [ ]:
# FIXED: Don't specify task_type for MLM, or let PEFT infer it
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model = model.to(device)

model.print_trainable_parameters()

## 4. Gender Token Masking Function (NEW)

In [ ]:
def get_gender_token_ids(tokenizer):
    """Get token IDs for common gender-specific words."""
    gender_words = ['he', 'she', 'his', 'her', 'him', 'himself', 'herself', 'man', 'woman', 'boy', 'girl']
    gender_token_ids = set()
    
    for word in gender_words:
        tokens = tokenizer.encode(word, add_special_tokens=False)
        gender_token_ids.update(tokens)
    
    return gender_token_ids

def mask_gender_tokens(input_ids, tokenizer, gender_token_ids):
    """
    Mask gender-specific tokens and create appropriate labels.
    
    Args:
        input_ids: Input token IDs [batch_size, seq_len]
        tokenizer: HuggingFace tokenizer
        gender_token_ids: Set of gender token IDs to mask
    
    Returns:
        masked_input_ids: Input with gender tokens replaced by [MASK]
        labels: Labels with -100 for non-gender tokens
    """
    labels = input_ids.clone()
    masked_input_ids = input_ids.clone()
    
    # Create a mask for gender tokens
    gender_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for token_id in gender_token_ids:
        gender_mask |= (input_ids == token_id)
    
    # Mask gender tokens in input
    masked_input_ids[gender_mask] = tokenizer.mask_token_id
    
    # Set non-gender tokens to -100 in labels (ignore in loss)
    labels[~gender_mask] = -100
    
    return masked_input_ids, labels

# Get gender token IDs
gender_token_ids = get_gender_token_ids(tokenizer)
print(f"Gender token IDs: {gender_token_ids}")
print(f"Total gender tokens: {len(gender_token_ids)}")

## 5. Fixed Debiasing Loss Function

In [ ]:
def debiasing_loss_fn(pro_logits, anti_logits, pro_labels, anti_labels, alpha=0.5):
    """
    Compute debiasing loss for MLM.
    
    Args:
        pro_logits: Logits for pro-stereotyped sentences [batch, seq_len, vocab_size]
        anti_logits: Logits for anti-stereotyped sentences [batch, seq_len, vocab_size]
        pro_labels: Labels for pro (only gender tokens, others -100)
        anti_labels: Labels for anti (only gender tokens, others -100)
        alpha: Balance between task loss and consistency loss
    
    Returns:
        Dictionary with loss components
    """
    # 1. MLM Task Loss (only on gender tokens)
    pro_logits_flat = pro_logits.view(-1, pro_logits.size(-1))
    anti_logits_flat = anti_logits.view(-1, anti_logits.size(-1))
    pro_labels_flat = pro_labels.view(-1)
    anti_labels_flat = anti_labels.view(-1)
    
    pro_task_loss = F.cross_entropy(pro_logits_flat, pro_labels_flat, ignore_index=-100)
    anti_task_loss = F.cross_entropy(anti_logits_flat, anti_labels_flat, ignore_index=-100)
    task_loss = (pro_task_loss + anti_task_loss) / 2
    
    # 2. Consistency Loss (predictions should be similar for pro and anti)
    # Create mask for positions where labels != -100
    mask = (pro_labels != -100).float().unsqueeze(-1)  # [batch, seq_len, 1]
    
    # Get probabilities
    pro_probs = F.softmax(pro_logits, dim=-1)
    anti_probs = F.softmax(anti_logits, dim=-1)
    
    # Only compute consistency on masked positions
    masked_pro_probs = pro_probs * mask
    masked_anti_probs = anti_probs * mask
    
    # MSE or KL divergence between distributions
    consistency_loss = F.mse_loss(masked_pro_probs, masked_anti_probs)
    
    # Alternative: KL divergence
    # consistency_loss = F.kl_div(
    #     F.log_softmax(pro_logits, dim=-1), 
    #     F.softmax(anti_logits, dim=-1), 
    #     reduction='batchmean'
    # )
    
    # 3. Total Loss
    total_loss = (1 - alpha) * task_loss + alpha * consistency_loss
    
    return {
        'total_loss': total_loss,
        'task_loss': task_loss,
        'consistency_loss': consistency_loss,
        'alpha': alpha
    }

## 6. Dataset Class (Updated)

In [ ]:
class CounterfactualDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        pro_text = item['pro']
        anti_text = item['anti']

        pro_encoding = self.tokenizer(
            pro_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        anti_encoding = self.tokenizer(
            anti_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'pro_input_ids': pro_encoding['input_ids'].squeeze(),
            'pro_attention_mask': pro_encoding['attention_mask'].squeeze(),
            'anti_input_ids': anti_encoding['input_ids'].squeeze(),
            'anti_attention_mask': anti_encoding['attention_mask'].squeeze()
        }

## 7. Create DataLoaders

In [ ]:
# Split dataset
train, test = train_test_split(data, test_size=0.2, random_state=42)

train_dataset = CounterfactualDataset(train, tokenizer)
test_dataset = CounterfactualDataset(test, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 8. Training Setup

In [ ]:
epochs = 3
optimizer = AdamW(model.parameters(), lr=5e-5)
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

print(f"Total training steps: {total_steps}")

## 9. Training Loop (FIXED)

In [ ]:
training_history = {
    'total_loss': [],
    'task_loss': [],
    'consistency_loss': []
}

for epoch in range(epochs):
    model.train()
    epoch_losses = {'total': 0.0, 'task': 0.0, 'consistency': 0.0}
    
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
        pro_input_ids = batch['pro_input_ids'].to(device)
        pro_attention_mask = batch['pro_attention_mask'].to(device)
        anti_input_ids = batch['anti_input_ids'].to(device)
        anti_attention_mask = batch['anti_attention_mask'].to(device)

        # FIXED: Mask gender tokens and create proper labels
        pro_masked_ids, pro_labels = mask_gender_tokens(pro_input_ids, tokenizer, gender_token_ids)
        anti_masked_ids, anti_labels = mask_gender_tokens(anti_input_ids, tokenizer, gender_token_ids)

        # Forward pass with MASKED inputs
        pro_outputs = model(input_ids=pro_masked_ids, attention_mask=pro_attention_mask)
        pro_logits = pro_outputs.logits

        anti_outputs = model(input_ids=anti_masked_ids, attention_mask=anti_attention_mask)
        anti_logits = anti_outputs.logits

        # Compute debiasing loss
        losses = debiasing_loss_fn(pro_logits, anti_logits, pro_labels, anti_labels, alpha=0.5)
        total_loss = losses['total_loss']

        # Backpropagation
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_losses['total'] += total_loss.item()
        epoch_losses['task'] += losses['task_loss'].item()
        epoch_losses['consistency'] += losses['consistency_loss'].item()

    # Calculate averages
    avg_total = epoch_losses['total'] / len(train_dataloader)
    avg_task = epoch_losses['task'] / len(train_dataloader)
    avg_consistency = epoch_losses['consistency'] / len(train_dataloader)
    
    training_history['total_loss'].append(avg_total)
    training_history['task_loss'].append(avg_task)
    training_history['consistency_loss'].append(avg_consistency)
    
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Total Loss: {avg_total:.4f}")
    print(f"  Task Loss: {avg_task:.4f}")
    print(f"  Consistency Loss: {avg_consistency:.4f}")

## 10. Visualize Training

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(training_history['total_loss'])
plt.title('Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 2)
plt.plot(training_history['task_loss'])
plt.title('Task Loss (MLM)')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 3)
plt.plot(training_history['consistency_loss'])
plt.title('Consistency Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

## 11. Save Model

In [ ]:
model_path = "output/debiased_model_fixed"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to {model_path}")

## 12. Evaluation

In [ ]:
from peft import PeftModel

# Load model for evaluation
base_model_test = AutoModelForMaskedLM.from_pretrained(model_name)
debiased_model = PeftModel.from_pretrained(base_model_test, model_path)
debiased_model = debiased_model.to(device)
debiased_model.eval()

# Evaluate
test_losses = {'total': 0.0, 'task': 0.0, 'consistency': 0.0}

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Evaluating"):
        pro_input_ids = batch['pro_input_ids'].to(device)
        pro_attention_mask = batch['pro_attention_mask'].to(device)
        anti_input_ids = batch['anti_input_ids'].to(device)
        anti_attention_mask = batch['anti_attention_mask'].to(device)

        # Mask gender tokens
        pro_masked_ids, pro_labels = mask_gender_tokens(pro_input_ids, tokenizer, gender_token_ids)
        anti_masked_ids, anti_labels = mask_gender_tokens(anti_input_ids, tokenizer, gender_token_ids)

        # Forward pass
        pro_outputs = debiased_model(input_ids=pro_masked_ids, attention_mask=pro_attention_mask)
        anti_outputs = debiased_model(input_ids=anti_masked_ids, attention_mask=anti_attention_mask)

        # Compute loss
        losses = debiasing_loss_fn(
            pro_outputs.logits, anti_outputs.logits, 
            pro_labels, anti_labels, alpha=0.5
        )

        test_losses['total'] += losses['total_loss'].item()
        test_losses['task'] += losses['task_loss'].item()
        test_losses['consistency'] += losses['consistency_loss'].item()

avg_test_total = test_losses['total'] / len(test_dataloader)
avg_test_task = test_losses['task'] / len(test_dataloader)
avg_test_consistency = test_losses['consistency'] / len(test_dataloader)

print(f"\nTest Set Results:")
print(f"  Total Loss: {avg_test_total:.4f}")
print(f"  Task Loss: {avg_test_task:.4f}")
print(f"  Consistency Loss: {avg_test_consistency:.4f}")

## 13. Test Bias Reduction

In [ ]:
def test_gender_bias(model, tokenizer, test_sentence_template, device):
    """
    Test gender bias by comparing predictions for he/she.
    """
    model.eval()
    
    # Get token IDs
    he_id = tokenizer.encode('he', add_special_tokens=False)[0]
    she_id = tokenizer.encode('she', add_special_tokens=False)[0]
    
    # Test both genders
    sentences = [
        test_sentence_template.replace('[GENDER]', tokenizer.mask_token),
    ]
    
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        # Find mask position
        mask_pos = (inputs['input_ids'] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
        
        # Get probabilities for he/she
        probs = F.softmax(logits[0, mask_pos, :], dim=-1)
        he_prob = probs[0, he_id].item()
        she_prob = probs[0, she_id].item()
        
        print(f"\nSentence: {sent}")
        print(f"  P(he): {he_prob:.4f}")
        print(f"  P(she): {she_prob:.4f}")
        print(f"  Bias ratio (he/she): {he_prob/she_prob:.2f}")

# Test examples
test_sentences = [
    "The nurse said [GENDER] would be back soon.",
    "The engineer explained [GENDER] was working on the project.",
    "The teacher told the students [GENDER] was proud of them.",
    "The CEO announced [GENDER] would be stepping down."
]

print("=" * 50)
print("TESTING DEBIASED MODEL")
print("=" * 50)

for sent in test_sentences:
    test_gender_bias(debiased_model, tokenizer, sent, device)